<a href="https://colab.research.google.com/github/umarashraf-analytics/govshield-leakage-detector/blob/main/GovShield_Leakage_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

def generate_datasets():
    np.random.seed(42)
    random.seed(42)
    start_date = datetime(2026, 1, 1)

    # --- 1. PDS Data Generation ---
    pds_data = []
    fps_shops = [f"FPS-SHOP-{i:03d}" for i in range(1, 11)]
    for i in range(1500):
        card_id = f"RATIO-CARD-{random.randint(100000, 999999)}"
        family_size = random.choice([2, 3, 4, 5, 6])
        allocated = family_size * 5
        fps_shop = random.choice(fps_shops)
        tx_date = start_date + timedelta(days=random.randint(0, 90))

        # Injecting Leakage: High overrides with max withdrawal at specific shops
        if fps_shop in ["FPS-SHOP-003", "FPS-SHOP-007"] and random.random() < 0.35:
            biometric_status = "FAILED"
            override_used = "YES"
            withdrawn = allocated
        else:
            biometric_status = np.random.choice(["SUCCESS", "FAILED"], p=[0.92, 0.08])
            override_used = "YES" if biometric_status == "FAILED" and random.random() < 0.15 else "NO"
            withdrawn = allocated if (biometric_status == "SUCCESS" or override_used == "YES") else 0

        pds_data.append([card_id, family_size, allocated, withdrawn, biometric_status, override_used, fps_shop, tx_date.strftime('%Y-%m-%d')])

    df_pds = pd.DataFrame(pds_data, columns=['Ration_Card_ID', 'Family_Size', 'Grain_Allocated_KG', 'Grain_Withdrawn_KG', 'Biometric_Status', 'Manual_Override_Used', 'FPS_Shop_ID', 'Transaction_Date'])
    df_pds.to_csv('pds_transactions.csv', index=False)

    # --- 2. LPG Data Generation ---
    lpg_data = []
    for i in range(1000):
        consumer_id = f"LPG-CON-{random.randint(100000, 999999)}"
        ctype = np.random.choice(["DOMESTIC", "COMMERCIAL"], p=[0.88, 0.12])

        if ctype == "COMMERCIAL":
            bookings = random.randint(5, 25)
            subsidy = 0.0
        else:
            # Injecting Leakage: Domestic accounts diverting to commercial quantities
            bookings = random.randint(6, 12) if random.random() < 0.04 else random.choice([0, 1, 1, 2])
            subsidy = bookings * 300.0

        lpg_data.append([consumer_id, ctype, bookings, subsidy, random.choice(["560001", "560034", "560102"])])

    df_lpg = pd.DataFrame(lpg_data, columns=['Consumer_ID', 'Cylinder_Type', 'Monthly_Bookings', 'Subsidy_Credited_INR', 'Delivery_Pincode'])
    df_lpg.to_csv('lpg_transactions.csv', index=False)
    print("🎯 Success: Data files 'pds_transactions.csv' and 'lpg_transactions.csv' created!")

generate_datasets()

🎯 Success: Data files 'pds_transactions.csv' and 'lpg_transactions.csv' created!


In [2]:
def run_leakage_audit():
    df_pds = pd.read_csv('pds_transactions.csv')
    df_lpg = pd.read_csv('lpg_transactions.csv')

    print("=== PDS AUDIT REPORT ===")
    # Find shops where manual overrides are suspiciously high
    fps_analysis = df_pds.groupby('FPS_Shop_ID').agg(
        Total_Transactions=('Ration_Card_ID', 'count'),
        Manual_Overrides=('Manual_Override_Used', lambda x: (x == 'YES').sum())
    ).reset_index()
    fps_analysis['Override_Rate_%'] = round((fps_analysis['Manual_Overrides'] / fps_analysis['Total_Transactions']) * 100, 2)

    # Flag shops with greater than 20% manual bypass rate
    flagged_shops = fps_analysis[fps_analysis['Override_Rate_%'] > 20.0]
    print(f"🚨 Flagged {len(flagged_shops)} Fair Price Shops exceeding acceptable biometric bypass boundaries:")
    print(flagged_shops.to_string(index=False))

    print("\n=== LPG SUBSIDY AUDIT REPORT ===")
    # Flag domestic accounts booking more than 5 cylinders a month (highly likely commercial diversion)
    diverted_lpg = df_lpg[(df_lpg['Cylinder_Type'] == 'DOMESTIC') & (df_lpg['Monthly_Bookings'] > 5)]
    total_lpg_leakage = diverted_lpg['Subsidy_Credited_INR'].sum()

    print(f"🚨 Detected {len(diverted_lpg)} residential accounts diverting domestic cylinders to commercial entities.")
    print(f"💰 Total estimated financial leakage recovered: ₹{total_lpg_leakage:,}")

    # Save the flagged outputs for our dashboard later
    flagged_shops.to_csv('flagged_pds_shops.csv', index=False)
    diverted_lpg.to_csv('flagged_lpg_diversions.csv', index=False)

run_leakage_audit()

=== PDS AUDIT REPORT ===
🚨 Flagged 2 Fair Price Shops exceeding acceptable biometric bypass boundaries:
 FPS_Shop_ID  Total_Transactions  Manual_Overrides  Override_Rate_%
FPS-SHOP-003                 148                58            39.19
FPS-SHOP-007                 133                54            40.60

=== LPG SUBSIDY AUDIT REPORT ===
🚨 Detected 40 residential accounts diverting domestic cylinders to commercial entities.
💰 Total estimated financial leakage recovered: ₹102,000.0


In [4]:
%%writefile app.py
import streamlit as st
import pandas as pd

st.set_page_config(page_title="GovShield Dashboard", layout="wide")
st.title("🏛️ GovShield: Subsidy Leakage Tracking System")
st.markdown("Real-time anomaly monitoring for public welfare distribution channels.")

# Load analytics data
try:
    pds_df = pd.read_csv('flagged_pds_shops.csv')
    lpg_df = pd.read_csv('flagged_lpg_diversions.csv')
except:
    st.error("Please run the backend analytical processing scripts first to compile data tables.")

# Metrics Highlights
col1, col2 = st.columns(2)
with col1:
    st.metric(label="Flagged High-Risk PDS Centers", value=f"{len(pds_df)} Shops")
with col2:
    st.metric(label="Recovered LPG Revenue Leakage", value=f"₹{lpg_df['Subsidy_Credited_INR'].sum():,}")

st.write("---")

# Layout Components
tab1, tab2 = st.columns(2)
with tab1:
    st.subheader("⚠️ High-Risk Ration Distribution Centers")
    st.dataframe(pds_df, use_container_width=True)
with tab2:
    st.subheader("🔥 Diverted Domestic LPG Connections")
    st.dataframe(lpg_df, use_container_width=True)

Overwriting app.py
